# AMLGuard: Anti-Money Laundering Detection under Extreme Class Imbalance

**Author:** Caio Bernardinelli  
**Date:** June 2026  
**Course:** Técnico em Inteligência Artificial — IFNMG  
**Objective:** Build an ML classifier to detect money laundering transactions
in the IBM AML dataset under extreme class imbalance (< 1% positive class).

---

## 1. Dataset Presentation

### Business Context

In 2025, the European Union centralized anti-money laundering enforcement
under the **AMLA (Authority for Anti-Money Laundering)**, operational since
July 2025 and based in Frankfurt. As of January 2026, AMLA assumed mandates
previously held by the EBA, and from 2028 will directly supervise ~40
high-risk cross-border institutions.

The core technical challenge: **illicit transactions represent less than 1%
of total volume** — a classic extreme class imbalance problem. Current systems
generate too many false positives, overwhelming compliance analysts.

### Dataset

**Source:** IBM Transactions for Anti-Money Laundering (AML)  
**Link:** https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml  
**License:** Community Data License Agreement - Sharing - Version 1.0  
**Version used:** HI-Small (High Illicit ratio — smallest version for rapid iteration)

### Variable Description

| Column | Type | Description |
|--------|------|-------------|
| Timestamp | object | Date and time of the transaction |
| From Bank | int64 | ID of the sending bank |
| Account | object | Account number of the sender |
| To Bank | int64 | ID of the receiving bank |
| Account.1 | object | Account number of the receiver |
| Amount Received | float64 | Amount received by the destination account |
| Receiving Currency | object | Currency of the received amount |
| Amount Paid | float64 | Amount paid by the source account |
| Payment Currency | object | Currency of the paid amount |
| Payment Format | object | Payment method (e.g. Cheque, Wire, Reinvestment) |
| Is Laundering | int64 | **Target variable** — 1 = illicit, 0 = legitimate |

---

In [1]:
# ============================================================================
# SETUP & IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')
import gdown

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc
)

# Imbalanced learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Explainability
import shap

# Global random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ All libraries imported successfully")
print(f"✓ Random state fixed: {RANDOM_STATE}")

✓ All libraries imported successfully
✓ Random state fixed: 42


In [5]:
# ============================================================================
# 2. DATA LOADING
# ============================================================================

print("="*80)
print("2. DATA LOADING")
print("="*80)

file_id = '1359N_tsRuUtCFMWCV6BtHjDdF280rb8e'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'HI-Small_Trans.csv', quiet=False)


df = pd.read_csv('HI-Small_Trans.csv')
print(df.shape)
print(df.columns.tolist())


# Basic inspection
print(f"\n✓ Dataset loaded successfully!")
print(f"\n{'─'*40}")
print(f"Shape:        {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"{'─'*40}")

print(f"\n--- First 10 rows ---")
print(df.head(10))

print(f"\n--- Data types ---")
print(df.dtypes)

print(f"\n--- Target variable (Is Laundering) ---")
print(df['Is Laundering'].value_counts())
print(f"\nIllicit transactions: {df['Is Laundering'].mean()*100:.4f}%")

2. DATA LOADING


Downloading...
From (original): https://drive.google.com/uc?id=1359N_tsRuUtCFMWCV6BtHjDdF280rb8e
From (redirected): https://drive.google.com/uc?id=1359N_tsRuUtCFMWCV6BtHjDdF280rb8e&confirm=t&uuid=028720aa-e986-4779-9563-fd5c0524b2a4
To: /content/HI-Small_Trans.csv
100%|██████████| 476M/476M [00:06<00:00, 74.9MB/s]


(5078345, 11)
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

✓ Dataset loaded successfully!

────────────────────────────────────────
Shape:        5,078,345 rows × 11 columns
Memory usage: 1890.96 MB
────────────────────────────────────────

--- First 10 rows ---
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:20         10  8000EBD30       10  8000EBD30   
1  2022/09/01 00:20       3208  8000F4580        1  8000F5340   
2  2022/09/01 00:00       3209  8000F4670     3209  8000F4670   
3  2022/09/01 00:02         12  8000F5030       12  8000F5030   
4  2022/09/01 00:06         10  8000F5200       10  8000F5200   
5  2022/09/01 00:03          1  8000F5AD0        1  8000F5AD0   
6  2022/09/01 00:08          1  8000EBAC0        1  8000EBAC0   
7  2022/09/01 00:16          1  8000EC1E0        1  8000EC1E0   
8  2022/09/01 00:26    